# NetSciX 2027 · Notebook 2 — From candidate locations to hypothetical spatial routes

**Workshop:** *From Incomplete Spatial Data to Networks: A Hands-on Tutorial in Predictive Modeling and Routing*  
**Javier M. Vera Zúñiga & Dina S. Cornejo Meza**

**Research question:** How do spatially ranked candidate locations become a *hypothetical* network, and how do different travel-cost assumptions change its routes?

**Core distinctions:**
- NB1 out-of-fold scores distinguish previously **recorded cells** from **unlabelled** cells. They are **not** probabilities that a cell contains an archaeological site.
- The selected nodes below are **computational candidates**, not known archaeological sites and not the 24 environmental attractors of the original corridor model.
- A least-cost path minimizes **our stated cost function**; it is not evidence that an ancient route existed there.
- Real Cañete grids, RF rankings and derived paths are **PRIVATE** pending heritage-data clearance. A fully synthetic mode is supplied for public teaching.

## 0. Run mode and dependencies

`DATA_MODE="demo"` is the default, public, self-contained exercise. Choosing `DATA_MODE="real"` requests the two private CSVs from the prior Notebook 1 and its private handoff. Switch to `"demo"` for a stand-alone **synthetic** exercise. No original archaeological locations are distributed in the public repo.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra, connected_components
from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

DATA_MODE = os.environ.get("NETSCIX_DATA_MODE", "demo")  # "real" or "demo"
SEED = 2027
CELL_SPACING_M = 250
NUMBER_OF_NODES = 12
MIN_NODE_SEPARATION_M = 3000
K_NEAREST_CANDIDATES = 2
EXPORT_RESULTS = False # Enable manually only for controlled handoff to Notebook 3
assert DATA_MODE in ("real", "demo")
print('Run mode:', DATA_MODE)

## 1. Load Notebook 1 ranking + environmental grid

In **real** mode, upload `canete_250m_PRIVATE.csv` and `nb1_oof_scores_PRIVATE.csv` together. The score file was generated using the same five spatial folds and RF specification as Notebook 1, and contains **out-of-fold** (rather than in-sample) ranking scores.

In **demo** mode, we construct a synthetic landscape and generate synthetic OOF scores with a small RF. All values, locations and labels are invented for training purposes.

In [ ]:
def synthetic_demo(seed=SEED):
    rng = np.random.default_rng(seed)
    xx, yy = np.meshgrid(np.arange(30), np.arange(26))
    x, y = (xx.ravel()*250.).astype(float), (yy.ravel()*250.).astype(float)
    z = 20 + .003*x + .001*y + 10*np.sin(x/1500)
    slope = np.clip(4 + 3*np.cos(y/900) + rng.normal(0,1.2,len(x)),0,None)
    river = np.abs(y - (3200 + 550*np.sin(x/1100)))
    canal = np.abs(x - 3400)
    latent = -.04*z - .2*slope - .0015*river + .0003*canal
    latent = (latent-latent.mean())/(latent.std()+1e-12)
    prob = .012 + .12/(1+np.exp(-latent))
    labels = rng.binomial(1,prob).astype(int)
    frame = pd.DataFrame(dict(cell_id=[f"SIM_{i:04d}" for i in range(len(x))], x=x, y=y,
       registered_point_cell=labels, altitud_m=z, pendiente_grados=slope,
       dist_acequia_m_aprox=canal, dist_rio_m_aprox=river))
    feats = ['altitud_m','pendiente_grados','dist_acequia_m_aprox','dist_rio_m_aprox']
    bx = np.floor((frame.x-frame.x.min())/3000).astype(int)
    by = np.floor((frame.y-frame.y.min())/3000).astype(int)
    grouping = bx.astype(str)+":"+by.astype(str)
    oof = np.full(len(frame),np.nan)
    for tr,te in GroupKFold(n_splits=5).split(frame[feats],labels,grouping):
        model=Pipeline([('impute',SimpleImputer(strategy='median')),
          ('rf',RandomForestClassifier(n_estimators=160,min_samples_leaf=6,
           max_features='sqrt',class_weight='balanced_subsample',n_jobs=-1,random_state=seed))])
        model.fit(frame.iloc[tr][feats],labels[tr])
        oof[te]=model.predict_proba(frame.iloc[te][feats])[:,1]
    frame['oof_rank_score']=oof
    assert frame.oof_rank_score.notna().all()
    return frame

if DATA_MODE == 'demo':
    df=synthetic_demo()
    print('SYNTHETIC grid and OOF ranking; same demo generation and fold/model specification as Notebook 1.')
else:
    candidates=[Path('/content'),Path.cwd(),Path('data/private'),Path('../data/private')]
    required=('canete_250m_PRIVATE.csv','nb1_oof_scores_PRIVATE.csv')
    def locate(name):
        env=os.getenv('NETSCIX_PRIVATE_DIR','')
        opts=([Path(env)/name] if env else [])+[p/name for p in candidates]
        return next((p for p in opts if p.is_file()),None)
    missing=[name for name in required if locate(name) is None]
    if missing:
        try:
            from google.colab import files
            print('Upload BOTH private input files (not for GitHub):',required)
            files.upload()
        except ImportError as e:
            raise FileNotFoundError('Provide private files under NETSCIX_PRIVATE_DIR, or switch DATA_MODE to demo.') from e
    paths={name:locate(name) for name in required}
    if any(v is None for v in paths.values()):
        raise FileNotFoundError('Missing private input(s): '+', '.join(k for k,v in paths.items() if v is None))
    base=pd.read_csv(paths[required[0]])
    oof=pd.read_csv(paths[required[1]])
    assert oof.cell_id.is_unique and base.cell_id.is_unique
    assert {'cell_id','oof_rank_score'}.issubset(oof.columns)
    df=base.merge(oof[['cell_id','oof_rank_score']],on='cell_id',how='left',validate='one_to_one')
    assert len(df)==len(base) and df.oof_rank_score.notna().all(), 'Score file must cover every grid cell'
    print('PRIVATE Cañete grid loaded. Do not publish grid, scores or site-derived locations.')


## 2. Select *candidate* nodes from **unlabelled** cells

**Rule fixed before viewing the resulting routes:** sort unlabelled cells by OOF score, choose up to 12, and enforce a minimum Euclidean distance of 3 km between selected nodes. Scores only rank cells relative to an incomplete recording inventory. We are **not** treating unlabelled cells as verified negatives or selected nodes as archaeological finds.

**Exercise:** adjust `MIN_NODE_SEPARATION_M` or `NUMBER_OF_NODES`, rerun selection, and discuss why the proposed graph changes. For cost-sensitivity comparisons below, keep these nodes fixed.

In [ ]:
pool = df[df.registered_point_cell.eq(0)].sort_values(['oof_rank_score','cell_id'],ascending=[False,True])
chosen=[]
for i in pool.index:
    pos=df.loc[i,['x','y']].to_numpy(dtype=float)
    if all(np.hypot(*(pos-df.loc[j,['x','y']].to_numpy(dtype=float))) >= MIN_NODE_SEPARATION_M for j in chosen):
        chosen.append(i)
    if len(chosen)>=NUMBER_OF_NODES: break
assert len(chosen)>=4,'Not enough spatially separated candidate nodes; reduce min distance.'
nodes=df.loc[chosen,['cell_id','x','y','oof_rank_score']].reset_index(drop=True).rename(columns={'cell_id':'node_cell_id'})
nodes.insert(0,'node_id',[f'N{i+1:02d}' for i in range(len(nodes))])
assert np.all(df.loc[chosen,'registered_point_cell'].eq(0))
print('Candidate nodes:',len(nodes),'(unlabelled, NOT archaeologically verified)')
print(nodes[['node_id','oof_rank_score']].round(4).to_string(index=False))

## 3. Fix candidate edges (same endpoints across scenarios)

Connect each candidate to its **two nearest candidate neighbors** in Euclidean space; deduplicate pairs. This is a simple workshop design choice, not an inferred archaeological interaction. Holding endpoints fixed lets us compare only changes to routing cost.

In [ ]:
D=cdist(nodes[['x','y']],nodes[['x','y']]);np.fill_diagonal(D,np.inf)
pairs=sorted({tuple(sorted((int(i),int(j)))) for i in range(len(nodes)) for j in np.argsort(D[i])[:K_NEAREST_CANDIDATES]})
print(len(pairs),'unique candidate pairs')
assert pairs

## 4. Build the 250 m movement grid

The **8-neighbor** graph respects real Euclidean movement distance (`250 m` orthogonal, approximately `353.6 m` diagonal). A diagonal edge is included **only if both adjacent orthogonal cells exist**, preventing shortcutting across holes in the study mask. We verify every candidate belongs to a connected graph.

In [ ]:
coord_to_idx={(int(round(x)),int(round(y))):i for i,(x,y) in enumerate(df[['x','y']].to_numpy())}
row=[];col=[];step=[]
for (x,y),i in coord_to_idx.items():
    for dx,dy in ((250,0),(0,250),(250,250),(250,-250)):
        p=(x+dx,y+dy)
        if p not in coord_to_idx: continue
        if dx and dy and ((x+dx,y) not in coord_to_idx or (x,y+dy) not in coord_to_idx): continue
        j=coord_to_idx[p]
        d=float(np.hypot(dx,dy))
        row.extend((i,j));col.extend((j,i));step.extend((d,d))
row=np.asarray(row,dtype=int);col=np.asarray(col,dtype=int);step=np.asarray(step,dtype=float)
graph_neighbor_keys=set(zip(row.tolist(),col.tolist()))
grid=csr_matrix((np.ones(len(row)),(row,col)),shape=(len(df),len(df)))
n_components,component=connected_components(grid,directed=False)
node_idx=np.array([coord_to_idx[(int(round(x)),int(round(y)))] for x,y in nodes[['x','y']].to_numpy()],dtype=int)
assert len(set(component[node_idx])) == 1, 'Candidates must belong to the same connected grid component'
print('Movement graph:',len(df),'cells,',len(row)//2,'undirected adjacencies,',n_components,'component(s)')

## 5. Define two **illustrative** cost scenarios

We transform slope and distance to the **modern** nearest river into nonnegative dimensionless penalties, clipped at their 95th percentile. `River-distance penalty` is *smaller near current mapped rivers*; this is just a test assumption, **not a claim about ancient rivers or travel choices**.

The cost to traverse neighboring cells is the physical step length times the mean of their cell penalties.

- Scenario A: **slope-dominant** (`slope weight = 2.0`, `river-distance weight = 0.4`)
- Scenario B: **river-distance-dominant** (`slope weight = 0.6`, `river-distance weight = 2.0`)

**Important:** this simplified teaching LCP differs from the original Cañete corridor model's environmental affinity, attractor ensemble and CA/k-shortest-path workflow. It is not claimed to reproduce the published model.

In [ ]:
def norm_positive(a):
    s=pd.Series(a,dtype='float64')
    s=s.fillna(s.median()).clip(lower=0).to_numpy()
    hi=np.percentile(s,95)
    return np.clip(s/max(hi,1e-9),0,1)
slope_penalty=norm_positive(df.pendiente_grados)
river_penalty=norm_positive(df.dist_rio_m_aprox)
SCENARIOS={'slope_dominant':(2.0,0.4),'river_distance_dominant':(0.6,2.0)}
def cell_cost(ws,wr):
    return 1.0+ws*slope_penalty+wr*river_penalty
for s,(ws,wr) in SCENARIOS.items():
    v=cell_cost(ws,wr)
    print(s,': cell cost range',f'{v.min():.2f}—{v.max():.2f}')

## 6. Least-Cost Path routing and route audit

For each scenario, Dijkstra's algorithm finds least-cost paths for our **fixed node pairs**. We audit that paths begin and end at the selected nodes and that all consecutive cells share a valid movement-grid edge. Weighted accumulated cost is **scenario-specific**, so compare route geometry and physical length across scenarios rather than treating cost totals as directly equivalent units.

In [ ]:
records=[];route_steps=[]
for scenario,(ws,wr) in SCENARIOS.items():
    cc=cell_cost(ws,wr)
    weights=step*(cc[row]+cc[col])*0.5
    wg=csr_matrix((weights,(row,col)),shape=(len(df),len(df)))
    origins=sorted({u for u,v in pairs})
    origins_idx=[node_idx[u] for u in origins]
    distances,predecessors=dijkstra(wg,directed=False,indices=origins_idx,return_predecessors=True)
    origin_to_k={u:k for k,u in enumerate(origins)}
    for u,v in pairs:
        start,target=int(node_idx[u]),int(node_idx[v])
        dist=float(distances[origin_to_k[u],target]);assert np.isfinite(dist)
        path=[target];cur=target
        while cur!=start:
            cur=int(predecessors[origin_to_k[u],cur]);assert cur>=0
            path.append(cur)
        path.reverse()
        coords=df[['x','y']].iloc[path].to_numpy()
        dphys=float(np.linalg.norm(np.diff(coords,axis=0),axis=1).sum())
        assert path[0]==start and path[-1]==target
        assert all((a,b) in graph_neighbor_keys for a,b in zip(path,path[1:])), 'Path uses an invalid grid connection'
        label_u,label_v=nodes.node_id.iloc[[u,v]]
        records.append(dict(scenario=scenario,source=label_u,target=label_v,physical_length_m=dphys,
               accumulated_model_cost=dist,number_of_cells=len(path)))
        for k,ix in enumerate(path):
            route_steps.append((scenario,label_u,label_v,k,df.cell_id.iloc[ix],float(df.x.iloc[ix]),float(df.y.iloc[ix])))
route_df=pd.DataFrame(records)
route_cells=pd.DataFrame(route_steps,columns=['scenario','source','target','step','cell_id','x','y'])
assert len(route_df)==2*len(pairs)
print(route_df.groupby('scenario').agg(n_routes=('source','size'),mean_length_m=('physical_length_m','mean'),mean_accumulated_cost=('accumulated_model_cost','mean')).round(2))

## 7. Compare changes in route geometry

This figure shows **hypothetical modeled paths only**. In private mode, the map can reveal sensitive information and **must not be copied into a public repository**.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,5),sharex=True,sharey=True)
for ax,(scenario,_) in zip(axs,SCENARIOS.items()):
    bg=df.sample(min(len(df),4000),random_state=SEED)
    ax.scatter(bg.x,bg.y,s=1,c='lightgray',alpha=.5,rasterized=True)
    part=route_cells[route_cells.scenario.eq(scenario)]
    for (_, _),g in part.groupby(['source','target'],sort=False):
        ax.plot(g.x,g.y,lw=1.5,alpha=.75,color='#2c7a7b')
    ax.scatter(nodes.x,nodes.y,s=30,c='#c65f27',zorder=10)
    ax.set_title(scenario.replace('_',' '))
    ax.set(xlabel='x (synthetic m)' if DATA_MODE=='demo' else 'UTM E (m)',
           ylabel='y (synthetic m)' if DATA_MODE=='demo' else 'UTM N (m)')
    ax.set_aspect('equal',adjustable='box')
fig.suptitle('SYNTHETIC tutorial routes' if DATA_MODE=='demo' else 'PRIVATE modeled Cañete routes — DO NOT PUBLISH')
plt.tight_layout();plt.show()

## 8. Same nodes and edges, different paths?

Summarize changes in physical route length and the overlap of traversed cells across cost scenarios. This is a descriptive **sensitivity comparison**, not field validation.

In [ ]:
names=list(SCENARIOS)
a=route_df[route_df.scenario.eq(names[0])].set_index(['source','target'])
b=route_df[route_df.scenario.eq(names[1])].set_index(['source','target'])
comparison=pd.DataFrame({'length_A_m':a.physical_length_m,'length_B_m':b.physical_length_m}).reset_index()
comparison['absolute_length_difference_m']=(comparison.length_A_m-comparison.length_B_m).abs()
sets={}
for (sc,u,v),g in route_cells.groupby(['scenario','source','target']):sets[(sc,u,v)]=set(g.cell_id)
comparison['cell_overlap_Jaccard']=[len(sets[(names[0],r.source,r.target)]&sets[(names[1],r.source,r.target)]) / max(1,len(sets[(names[0],r.source,r.target)]|sets[(names[1],r.source,r.target)])) for r in comparison.itertuples()]
print(comparison[['source','target','absolute_length_difference_m','cell_overlap_Jaccard']].round(3).to_string(index=False))
print('Average Jaccard overlap:',round(comparison.cell_overlap_Jaccard.mean(),3))

## 9. Optional **private** export for Notebook 3

`EXPORT_RESULTS` is **False by default**. To export, consciously set it to `True` in the initial configuration cell, then rerun this export cell. Exports include candidate location and route coordinates and therefore require private storage and authorization before disclosure. Never commit empirical exports to a public repository.

Notebook 3 will use `nb2_nodes_PRIVATE.csv` and `nb2_edges_PRIVATE.csv`. `nb2_route_cells_PRIVATE.csv` supports geographic visualizations but is not necessary for topological measures.

In [ ]:
if EXPORT_RESULTS:
    tag='PRIVATE' if DATA_MODE=='real' else 'SYNTHETIC'
    out=Path('/content') if Path('/content').is_dir() else Path.cwd()
    nodes.to_csv(out/f'nb2_nodes_{tag}.csv',index=False)
    route_df.to_csv(out/f'nb2_edges_{tag}.csv',index=False)
    route_cells.to_csv(out/f'nb2_route_cells_{tag}.csv',index=False)
    print('Exported locally under',out,'. Treat PRIVATE files as confidential.')
else:
    print('Export disabled (recommended until Notebook 3 handoff is needed).')

### Questions for discussion

- What changes if nodes are chosen by top-ranked scores compared with uniform geographic coverage? Who is missed by the historical survey inventory?
- How much does modern river proximity alter paths? Does this hypothetical proxy fit the past setting we are studying?
- What evidence would be needed to test routes independently?
- We held the node set and pairing rule fixed. What additional uncertainty appears when they are allowed to change?

**Notebook 3:** construct a graph whose vertices are selected **candidate nodes** and whose edges are **modeled** connections, then compare network properties without claiming that central nodes were historically important.

**Try it:** Compare the two scenarios using the route-overlap table. Which paths change even though the pairs of connected nodes remain constant? If you modify node-selection parameters, rerun all subsequent cells; do not compare cost scenarios with different node sets.